# Results overview

The figures and tables for Section 3.3, grouped by comparison case. Section 1 is Within Exact.
Sections 2 and 3 hold the ANN material until they are organised the same way.

- Run `python graph.py` first if the per-method figures need redrawing from `results/`
- Run all cells. Charts drawn here are saved to `figures/exact/` for the report
- The images are embedded in the output, so clear the outputs before a commit to keep the file small
- `show(paths, columns)` shows PNG files in a grid; `table(method, k)` loads one method's rows

In [ ]:
import base64
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd
from IPython.display import HTML, display

import graph

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
K = graph.REPORT_K
GREY = "#898781"  # reference lines, as in graph.py

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)


def show(paths, columns=2):
    """Show PNG files side by side, `columns` per row, each captioned with its file name."""
    paths = [Path(path) for path in paths]
    missing = [path for path in paths if not path.exists()]
    if missing:
        print("missing:", ", ".join(str(path) for path in missing))
    width = 100 / columns - 1
    cells = []
    for path in paths:
        if path.exists():
            data = base64.b64encode(path.read_bytes()).decode()
            cells.append(
                f'<figure style="flex:0 0 {width:.1f}%;margin:0 0 12px 0">'
                f'<img src="data:image/png;base64,{data}" style="width:100%;border:1px solid #ddd">'
                f'<figcaption style="font:12px monospace;color:#555;text-align:center">{path.name}</figcaption>'
                f'</figure>')
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:1%">{"".join(cells)}</div>'))


def figures(folder, *names):
    """Paths in figures/<folder>: the named files, or every PNG in the folder when no names are given."""
    folder = FIG_DIR / folder
    return [folder / name for name in names] if names else sorted(folder.glob("*.png"))


def table(method, k=K):
    """One method's result rows at k as a DataFrame (all k when k=None)."""
    return pd.DataFrame(graph.load_results(method, RESULTS_DIR, k))

## 1. Within Exact: linear scan, KD-tree, ball tree

All three methods return the true neighbours, so accuracy is 1.0 up to distance ties.
The comparison is about cost: query speed, build time and index size, and how they change
with `leaf_size`, with `k` and with the dimension of the data.

The trees have six `leaf_size` settings. Where one number per tree is needed, the fastest
`leaf_size` at k = 10 is used (the rule in `graph.best_rows`). Linear scan has no parameter.

In [ ]:
EXACT = {"linear_scan": "Linear scan", "kd_tree": "KD-tree", "ball_tree": "Ball tree"}
TREES = {"kd_tree": (1.0, "o"), "ball_tree": (0.5, "^")}  # shade of the dataset colour, marker
EXACT_DIR = FIG_DIR / "exact"
EXACT_DIR.mkdir(parents=True, exist_ok=True)

linear = {row["dataset"]: row for row in graph.load_results("linear_scan", RESULTS_DIR)}
all_k = {method: graph.load_results(method, RESULTS_DIR, k=None) for method in EXACT}
trees, best = {}, {}
for method in TREES:
    rows = graph.load_results(method, RESULTS_DIR)
    graph.add_speedup(rows, RESULTS_DIR)  # row["speedup"] = qps / linear scan qps on the same dataset
    trees[method] = rows
    best[method] = graph.best_rows(rows, 0.95)  # fastest leaf_size per dataset
LEAF_SIZES = sorted({row["leaf_size"] for row in trees["kd_tree"]})


def raw_mb(row):
    """Size of the raw float32 vectors in MB, the floor for every index size."""
    return row["n_base"] * row["dim"] * 4 / 1e6


def save_and_show(fig, name, folder=None):
    """Save a chart into figures/exact/ (or `folder`) and show it."""
    path = (folder or EXACT_DIR) / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200, facecolor="white")
    plt.close(fig)
    show([path], columns=1)
    return path


def dataset_panels(title, ylabel, xlabel):
    """One panel per dataset, titled with the dataset label; returns (fig, {dataset: ax})."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), layout="constrained")
    for ax, dataset in zip(axes, graph.DATASETS):
        ax.set_title(graph.DATASETS[dataset][0], loc="left", fontweight="bold")
        ax.set_xlabel(xlabel)
    axes[0].set_ylabel(ylabel)
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    return fig, dict(zip(graph.DATASETS, axes))


def method_legend(fig, linear_marker=None):
    """One legend beside the panels. Each panel uses its dataset's colour, so the legend is grey."""
    handles = [Line2D([], [], color=graph._shade("#3d3c39", weight), marker=marker, linewidth=2,
                      markersize=7, label=EXACT[method]) for method, (weight, marker) in TREES.items()]
    handles.append(Line2D([], [], color=GREY, linestyle="--", linewidth=1.2, marker=linear_marker,
                          markersize=6, label=EXACT["linear_scan"]))
    fig.legend(handles=handles, loc="outside right upper", fontsize=9)


def grouped_bars(values, ylabel, title, name, fmt, logy=False, hline=None):
    """values[method][dataset]: one group of bars per method, one bar per dataset in its colour."""
    fig, ax = plt.subplots(figsize=(8, 4.2), layout="constrained")
    width = 0.26
    for j, dataset in enumerate(graph.DATASETS):
        label, colour, _ = graph.DATASETS[dataset]
        heights = [values[method][dataset] for method in values]
        bars = ax.bar([i + (j - 1) * width for i in range(len(values))], heights, width=width - 0.03,
                      color=colour, label=label)
        ax.bar_label(bars, labels=[fmt.format(height) for height in heights], padding=2, fontsize=8)
    if hline:
        ax.axhline(hline[0], color=GREY, linestyle="--", linewidth=1.2, label=hline[1])
    ax.set_xticks(range(len(values)), [EXACT[method] for method in values])
    if logy:
        ax.set_yscale("log")
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.15)
    ax.set_title(title, loc="left", fontweight="bold")
    fig.legend(loc="outside right upper", fontsize=9)
    return save_and_show(fig, name)

### 1.1 Query speed: QPS of the three exact methods

Question: how fast does each exact method answer a query, and does any `leaf_size` let a tree beat the linear scan?
The dashed line is the linear scan on the same dataset; a tree line below it is slower than brute force.

In [ ]:
fig, axes = dataset_panels(f"Exact methods: QPS against leaf_size (k = {K})", "Queries per second", "leaf_size")
for dataset, ax in axes.items():
    colour = graph.DATASETS[dataset][1]
    for method, (weight, marker) in TREES.items():
        points = sorted((row["leaf_size"], row["qps"]) for row in trees[method] if row["dataset"] == dataset)
        ax.plot(*zip(*points), color=graph._shade(colour, weight), marker=marker, linewidth=2, markersize=7)
    ax.axhline(linear[dataset]["qps"], color=GREY, linestyle="--", linewidth=1.2)
    ax.set_xscale("log", base=2)
    ax.set_xticks(LEAF_SIZES, [str(leaf) for leaf in LEAF_SIZES])
    ax.minorticks_off()
    ax.set_yscale("log")
method_legend(fig)
save_and_show(fig, "exact_qps_vs_leaf_size.png")

In [ ]:
# The fastest setting of each tree against the linear scan, per dataset.
pd.DataFrame([{
    "dataset": dataset, "dim": linear[dataset]["dim"],
    "linear scan qps": linear[dataset]["qps"],
    "KD-tree qps": best["kd_tree"][dataset]["qps"], "KD-tree leaf_size": best["kd_tree"][dataset]["leaf_size"],
    "KD-tree speed-up": round(best["kd_tree"][dataset]["speedup"], 3),
    "ball tree qps": best["ball_tree"][dataset]["qps"], "ball tree leaf_size": best["ball_tree"][dataset]["leaf_size"],
    "ball tree speed-up": round(best["ball_tree"][dataset]["speedup"], 3),
} for dataset in graph.DATASETS]).set_index("dataset")

### 1.2 Query speed against k

Question: how does the number of neighbours per query change the cost of each exact method?
Each tree uses its fastest `leaf_size` from the table above. A tree must visit more leaves to collect more neighbours,
so its QPS falls slowly with k. The linear scan ranks every base vector whatever k is, yet its k = 1 point is
5 to 7 times faster than k = 10. The probable cause is in `methods/linear_scan.py`: at k = 1 the matrix multiply
dominates, and from k = 10 the per-block `argpartition` of the scores costs more than the distances themselves.
Check this with Kellie before the report explains it.

In [ ]:
fig, axes = dataset_panels(f"Exact methods: QPS against k (trees at their fastest leaf_size)",
                           "Queries per second", "k (neighbours per query)")
for dataset, ax in axes.items():
    colour = graph.DATASETS[dataset][1]
    for method, (weight, marker) in TREES.items():
        leaf = best[method][dataset]["leaf_size"]
        points = sorted((row["k"], row["qps"]) for row in all_k[method]
                        if row["dataset"] == dataset and row["leaf_size"] == leaf)
        ax.plot(*zip(*points), color=graph._shade(colour, weight), marker=marker, linewidth=2, markersize=7)
    points = sorted((row["k"], row["qps"]) for row in all_k["linear_scan"] if row["dataset"] == dataset)
    ax.plot(*zip(*points), color=GREY, linestyle="--", marker="s", linewidth=1.5, markersize=6)
    ks = sorted({row["k"] for row in all_k["linear_scan"]})
    ax.set_xscale("log")
    ax.set_xticks(ks, [str(k) for k in ks])
    ax.minorticks_off()
    ax.set_yscale("log")
method_legend(fig, linear_marker="s")
save_and_show(fig, "exact_qps_vs_k.png")

In [ ]:
# Query time in ms per query at every k, trees at their fastest leaf_size.
chosen = [row for row in all_k["linear_scan"]] + [
    row for method in TREES for row in all_k[method]
    if row["leaf_size"] == best[method][row["dataset"]]["leaf_size"]]
pd.DataFrame(chosen).pivot_table(index=["dataset", "method"], columns="k", values="query_time_ms")

### 1.3 Build time

Question: what does each exact method spend before the first query?
The linear scan only checks the data and precomputes the squared norms. A tree splits the data into nodes.
Trees use their fastest `leaf_size`; the y axis is logarithmic because the values span four orders of magnitude.

In [ ]:
build = {"linear_scan": {dataset: linear[dataset]["build_time_s"] for dataset in graph.DATASETS}}
for method in TREES:
    build[method] = {dataset: best[method][dataset]["build_time_s"] for dataset in graph.DATASETS}
grouped_bars(build, "Build time (s)", "Exact methods: build time",
             "exact_build_time.png", "{:.3g}", logy=True)

### 1.4 Index size

Question: how much memory does each exact method need to answer queries, compared with the raw vectors?
The value is the index size divided by the size of the raw float32 vectors, so the three datasets share one axis.
The linear scan keeps the vectors plus one norm per vector, so it sits at the floor of 1.0. The excess of a tree is its node and bound arrays.

In [ ]:
overhead = {"linear_scan": {dataset: linear[dataset]["index_size_mb"] / raw_mb(linear[dataset])
                            for dataset in graph.DATASETS}}
for method in TREES:
    overhead[method] = {dataset: best[method][dataset]["index_size_mb"] / raw_mb(best[method][dataset])
                        for dataset in graph.DATASETS}
grouped_bars(overhead, "Index size / raw data size", "Exact methods: index overhead",
             "exact_index_overhead.png", "{:.2f}", hline=(1.0, "raw data"))

In [ ]:
# Build time and index size in MB per method and dataset, trees at their fastest leaf_size.
pd.DataFrame([{"dataset": dataset, "method": EXACT[method], "leaf_size": row.get("leaf_size"),
               "build_time_s": row["build_time_s"], "index_size_mb": row["index_size_mb"],
               "raw_data_mb": round(raw_mb(row), 1)}
              for dataset in graph.DATASETS
              for method, row in [("linear_scan", linear[dataset])] + [(m, best[m][dataset]) for m in TREES]]
             ).set_index(["dataset", "method"])

### 1.5 Accuracy: all three are exact

Question: do the exact methods really return the true neighbours at every k?
No chart is needed; the table shows recall, NDCG and distance ratio at k = 1, 10 and 100.
Values below 1.0 on SIFT1M and GIST1M come from distance ties: a few queries have two base vectors at the same distance,
and the ground truth lists one of them while the method returns the other. The distance ratio stays at exactly 1.0.

In [ ]:
pd.DataFrame(chosen).pivot_table(index=["dataset", "method"], columns="k",
                                 values=["recall", "ndcg", "distance_ratio"])

### 1.6 Effect of dimension

Question: how does the dimension of the data change the trees' advantage over the linear scan?
Each point is the best speed-up of a tree on one dataset, plotted at that dataset's dimension.
A value below 1.0 means the tree is slower than brute force.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4), layout="constrained")
dims = {dataset: linear[dataset]["dim"] for dataset in graph.DATASETS}
for method, (weight, marker) in TREES.items():
    points = sorted((dims[dataset], best[method][dataset]["speedup"]) for dataset in graph.DATASETS)
    ax.plot(*zip(*points), color=graph._shade("#3d3c39", weight), marker=marker, linewidth=2, markersize=7,
            label=EXACT[method])
ax.axhline(1.0, color=GREY, linestyle="--", linewidth=1.2, label=EXACT["linear_scan"])
ax.set_xscale("log")
ax.set_xticks(sorted(dims.values()), [f"{graph.DATASETS[d][0].split(' (')[0]}\n{dims[d]}-d"
                                      for d in sorted(dims, key=dims.get)])
ax.minorticks_off()
ax.set_yscale("log")
ax.set_xlabel("Dimension of the dataset")
ax.set_ylabel("Best speed-up over linear scan (x)")
ax.set_title(f"Exact trees: best speed-up against dimension (k = {K})", loc="left", fontweight="bold")
fig.legend(loc="outside right upper", fontsize=9)
save_and_show(fig, "exact_speedup_vs_dim.png")

### 1.7 Parameter study: leaf_size

The per-tree charts from `graph.py`, for the parameter settings part of the report.
Left: KD-tree against ball tree on each dataset. Below: each tree's own speed-up and best-setting charts.

In [ ]:
show(figures("trees"), columns=1)
show(figures("kd_tree", "kd_tree_speedup_vs_leaf_size.png") + figures("ball_tree", "ball_tree_speedup_vs_leaf_size.png"))
show(figures("kd_tree", "kd_tree_best.png") + figures("ball_tree", "ball_tree_best.png"), columns=1)

In [ ]:
# QPS of both trees by leaf_size, one column per dataset and tree.
pd.concat([table("kd_tree"), table("ball_tree")]).pivot_table(index="leaf_size", columns=["dataset", "method"], values="qps")

## 2. Within ANN: Annoy, IVF, PQ, IVF-PQ

Each approximate method trades accuracy for speed through its parameters, so the methods are compared
in two ways: the whole trade-off curve, and one matched-accuracy setting per method.
The matched setting is the fastest one with recall@10 of at least 0.95 (`TARGET`). When a method never
reaches the target on a dataset, its most accurate setting is shown instead and its bar is hatched.
PQ counts asymmetric search only. `results/ivf_pq.csv` is not in the repository yet, so IVF-PQ is skipped
until it arrives; the code picks it up automatically.

In [ ]:
from matplotlib.patches import Patch

# Exact methods in greys, approximate methods in colours: label, colour, marker.
METHODS = {
    "linear_scan": ("Linear scan", "#3d3c39", "s"),
    "kd_tree": ("KD-tree", "#6f6e6a", "o"),
    "ball_tree": ("Ball tree", "#a3a29d", "^"),
    "annoy": ("Annoy", "#6a3fb5", "o"),
    "ivf": ("IVF", "#1a9c8f", "s"),
    "pq": ("PQ", "#d6529c", "^"),
    "ivf_pq": ("IVF-PQ", "#8c6d1f", "D"),
}
TARGET = 0.95
ANN_DIR = FIG_DIR / "ann"
COMPARISON_DIR = FIG_DIR / "comparison"
FIXED = {"dataset", "method", "n_base", "n_queries", "dim", "k", "build_time_s", "index_size_mb",
         "query_time_ms", "qps", "recall", "ndcg", "distance_ratio", "speedup"}

ann = {}
for method in ("annoy", "ivf", "pq", "ivf_pq"):
    rows = graph.load_results(method, RESULTS_DIR)
    if method == "pq":
        rows = [row for row in rows if row["symmetric"] != "True"]
    if rows:
        graph.add_speedup(rows, RESULTS_DIR)
        ann[method] = rows
ANN = list(ann)
print("ANN methods with results:", ANN)

# The chosen setting per method and dataset: exact methods at their fastest leaf_size, ANN at the target.
chosen_setting = {"linear_scan": linear, **best}
for method in ANN:
    chosen_setting[method] = graph.best_rows(ann[method], TARGET)


def setting_of(row):
    """The parameter values of a result row as one short string."""
    return ", ".join(f"{key}={value}" for key, value in row.items() if key not in FIXED)


def reached(row):
    return row["recall"] >= TARGET


def pareto(points):
    """The (recall, qps) points that no other point beats on both, sorted by recall."""
    front = []
    for recall, qps in sorted(points, reverse=True):  # highest recall first
        if not front or qps > front[-1][1]:
            front.append((recall, qps))
    return sorted(front)


def best_table(methods):
    """The chosen setting of each method per dataset, with its measures."""
    rows = []
    for method in methods:
        for dataset, row in chosen_setting[method].items():
            rows.append({"dataset": dataset, "method": METHODS[method][0], "setting": setting_of(row),
                         "reaches_target": reached(row), "qps": row["qps"], "speedup": round(row.get("speedup", 1.0), 3),
                         "recall": row["recall"], "recall_loss": round(1 - row["recall"], 4), "ndcg": row["ndcg"],
                         "distance_ratio": row["distance_ratio"], "build_time_s": row["build_time_s"],
                         "index_size_mb": row["index_size_mb"]})
    return (pd.DataFrame(rows).sort_values(["dataset", "qps"], ascending=[True, False])
            .set_index(["dataset", "method"]))


def method_bars(methods, metric, ylabel, title, name, folder, fmt, logy=False, hline=None):
    """One group of bars per dataset, one bar per method at its chosen setting; hatched when below TARGET."""
    fig, ax = plt.subplots(figsize=(10, 4.4), layout="constrained")
    width = 0.8 / len(methods)
    for j, method in enumerate(methods):
        label, colour, _ = METHODS[method]
        xs, heights, hatches = [], [], []
        for i, dataset in enumerate(graph.DATASETS):
            row = chosen_setting[method].get(dataset)
            if row is None:
                continue
            xs.append(i + (j - (len(methods) - 1) / 2) * width)
            heights.append(row[metric])
            hatches.append("" if reached(row) else "///")
        bars = ax.bar(xs, heights, width=width - 0.02, color=colour, label=label, edgecolor="#333333", linewidth=0.4)
        for bar, hatch in zip(bars, hatches):
            bar.set_hatch(hatch)
        ax.bar_label(bars, labels=[fmt.format(height) for height in heights], padding=2, fontsize=7)
    if hline:
        ax.axhline(hline[0], color=GREY, linestyle="--", linewidth=1.2, label=hline[1])
    ax.set_xticks(range(len(graph.DATASETS)), [graph.DATASETS[dataset][0] for dataset in graph.DATASETS])
    if logy:
        ax.set_yscale("log")
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.15)
    ax.set_title(title, loc="left", fontweight="bold")
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor="white", edgecolor="#333333", hatch="///", label=f"below recall {TARGET}"))
    fig.legend(handles=handles, loc="outside right upper", fontsize=9)
    return save_and_show(fig, name, folder)

### 2.1 Speed against accuracy: the whole trade-off

Question: at any accuracy level, which method answers queries fastest?
Every small point is one parameter setting; the line joins the settings that no other setting of the same method beats
on both recall and QPS. The dashed line is the linear scan, the dotted line the recall target.

In [ ]:
fig, axes = dataset_panels(f"ANN methods: speed against accuracy (k = {K})", "Queries per second", f"Recall@{K}")
for dataset, ax in axes.items():
    for method in ANN:
        points = [(row["recall"], row["qps"]) for row in ann[method] if row["dataset"] == dataset]
        if not points:
            continue
        label, colour, marker = METHODS[method]
        ax.scatter(*zip(*points), color=colour, marker=marker, s=16, alpha=0.3)
        ax.plot(*zip(*pareto(points)), color=colour, marker=marker, linewidth=2, markersize=6, label=label)
    ax.axhline(linear[dataset]["qps"], color=GREY, linestyle="--", linewidth=1.2, label="Linear scan")
    ax.axvline(TARGET, color=GREY, linestyle=":", linewidth=1.2, label=f"recall {TARGET}")
    ax.set_yscale("log")
    ax.set_xlim(0, 1.02)
handles, labels = next(iter(axes.values())).get_legend_handles_labels()
fig.legend(handles, labels, loc="outside right upper", fontsize=9)
save_and_show(fig, "ann_recall_vs_qps.png", ANN_DIR)

### 2.2 Query speed at matched accuracy

Question: when every method must reach recall@10 of 0.95, which is fastest, and which cannot get there at all?

In [ ]:
method_bars(ANN, "qps", "Queries per second", f"ANN methods: QPS at recall@{K} >= {TARGET}",
            "ann_qps_at_target.png", ANN_DIR, "{:,.0f}", logy=True)
best_table(ANN)

### 2.3 Memory against accuracy

Question: how much memory does each method need, and what does compression cost in accuracy?
Annoy and IVF keep the full vectors, so their size barely moves with the parameters. PQ keeps only short codes,
so its size and its accuracy both grow with the code length. The dashed line is the raw data size.

In [ ]:
fig, axes = dataset_panels(f"ANN methods: index size against accuracy (k = {K})", "Index size (MB)", f"Recall@{K}")
for dataset, ax in axes.items():
    for method in ANN:
        points = [(row["recall"], row["index_size_mb"]) for row in ann[method] if row["dataset"] == dataset]
        if not points:
            continue
        label, colour, marker = METHODS[method]
        ax.scatter(*zip(*points), color=colour, marker=marker, s=28, alpha=0.75, label=label)
    ax.axhline(raw_mb(linear[dataset]), color=GREY, linestyle="--", linewidth=1.2, label="raw data")
    ax.axvline(TARGET, color=GREY, linestyle=":", linewidth=1.2, label=f"recall {TARGET}")
    ax.set_yscale("log")
    ax.set_xlim(0, 1.02)
handles, labels = next(iter(axes.values())).get_legend_handles_labels()
fig.legend(handles, labels, loc="outside right upper", fontsize=9)
save_and_show(fig, "ann_recall_vs_size.png", ANN_DIR)
method_bars(ANN, "index_size_mb", "Index size (MB)", f"ANN methods: index size at recall@{K} >= {TARGET}",
            "ann_index_size_at_target.png", ANN_DIR, "{:,.0f}", logy=True)

### 2.4 Build time at matched accuracy

Question: what does each method spend before the first query to reach the target?
IVF trains k-means over nlist centres, PQ trains one codebook per sub-vector, Annoy builds n_trees trees.

In [ ]:
method_bars(ANN, "build_time_s", "Build time (s)", f"ANN methods: build time at recall@{K} >= {TARGET}",
            "ann_build_time_at_target.png", ANN_DIR, "{:.3g}", logy=True)

### 2.5 Parameter study per method

The per-method figures from `graph.py`, for the parameter settings part of the report, with each table at k = 10.

In [ ]:
show(figures("annoy"), columns=2)
table("annoy")

In [ ]:
IVF_METRICS = ["recall", "ndcg", "distance_ratio", "qps", "query_time_ms", "build_time_s"]
show([FIG_DIR / "ivf" / f"ivf_{metric}_vs_nprobe.png" for metric in IVF_METRICS], columns=3)
show([FIG_DIR / "ivf" / f"ivf_{metric}_vs_nlist.png" for metric in IVF_METRICS], columns=3)
show(figures("ivf", "ivf_tradeoff.png", "ivf_best.png"), columns=1)
table("ivf")

In [ ]:
PQ_METRICS = ["recall", "ndcg", "distance_ratio", "qps", "query_time_ms", "build_time_s", "index_size_mb"]
show([FIG_DIR / "pq" / f"pq_{metric}_vs_m.png" for metric in PQ_METRICS], columns=2)
show(figures("pq", "pq_symmetric_vs_asymmetric.png", "pq_tradeoff_qps.png", "pq_tradeoff_size.png", "pq_best.png"),
     columns=1)
table("pq")

In [ ]:
show(figures("ivf_pq"), columns=3)
table("ivf_pq")

## 3. Exact against ANN: accuracy, query time, build time

The linear scan is the reference at recall 1.0. Each other method is shown at its chosen setting:
the exact trees at their fastest `leaf_size`, the ANN methods at the recall target from Section 2.
The question is what each method pays and gains against brute force.
The trade-off chart in 2.1 already places the linear scan among the ANN curves; this section adds the trees
and compares one setting per method.

### 3.1 Query speed: speed-up over the linear scan

Question: how much faster than brute force is each method at its chosen setting?
Values below the dashed line are slower than the linear scan.

In [ ]:
ALL_METHODS = ["kd_tree", "ball_tree"] + ANN
method_bars(ALL_METHODS, "speedup", "Speed-up over linear scan (x)",
            f"Speed-up over linear scan at the chosen setting (k = {K})",
            "comparison_speedup.png", COMPARISON_DIR, "{:.3g}x", logy=True, hline=(1.0, "Linear scan"))

### 3.2 Accuracy: what the speed costs

Question: how many true neighbours does each method lose at its chosen setting, and how far away are its answers?
Exact methods lose nothing beyond ties. The distance ratio shows that the neighbours IVF misses are almost as close
as the true ones, while PQ's are visibly further away.

In [ ]:
method_bars(["linear_scan"] + ALL_METHODS, "recall", f"Recall@{K}",
            f"Recall@{K} at the chosen setting", "comparison_recall.png", COMPARISON_DIR, "{:.3f}")
method_bars(["linear_scan"] + ALL_METHODS, "distance_ratio", f"Distance ratio@{K}",
            f"Distance ratio@{K} at the chosen setting (1.0 is exact)", "comparison_distance_ratio.png",
            COMPARISON_DIR, "{:.3f}")

### 3.3 Build time: what the speed costs up front

Question: how long does each method take to get ready, against the linear scan's one pass over the data?

In [ ]:
method_bars(["linear_scan"] + ALL_METHODS, "build_time_s", "Build time (s)",
            "Build time at the chosen setting", "comparison_build_time.png", COMPARISON_DIR, "{:.3g}", logy=True)

### 3.4 Summary table

Every method at its chosen setting, per dataset, sorted by QPS. `recall_loss` is 1 - recall.
`reaches_target` is False when the method never reaches the recall target and its most accurate setting is shown.

In [ ]:
best_table(["linear_scan"] + ALL_METHODS)